# Build and publish the final Korean Olympiad TDCS dataset

This notebook creates the final balanced dataset from the 5,000 entropy candidates:

1. Assign D1–D5 independently inside every problem cluster using entropy quintiles.
2. Remove rows at or above 95% fuzzy similarity to OlympiadBench-Math-Ko.
3. Select exactly 33 rows from each of 20 clusters × 5 difficulty levels.
4. Put 30 rows per cell in `train` and 3 rows per cell in `validation`.
5. Save exactly 3,000 train and 300 validation examples to Drive and upload a DatasetDict to Hugging Face.

In [ ]:
%pip install -q -U datasets huggingface_hub pandas pyarrow rapidfuzz tqdm

In [ ]:
from google.colab import drive, userdata

drive.mount("/content/drive")
HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("Add a Hugging Face write token named HF_TOKEN to Colab Secrets")
print("HF_TOKEN loaded from Colab Secrets")

In [ ]:
# ---- User controls ----
CANDIDATES_PATH = (
    "/content/drive/MyDrive/Korean-TDCS/data/olympiad_entropy/"
    "entropy_candidates_250_per_cluster.parquet"
)
ENTROPY_METRICS_PATH = (
    "/content/drive/MyDrive/Korean-TDCS/data/olympiad_entropy/entropy_metrics.csv"
)
OUTPUT_ROOT = "/content/drive/MyDrive/Korean-TDCS/data/final_olympiad_tdcs_3k"
BENCHMARK_DATASET_ID = "ChuGyouk/OlympiadBench-Math-Ko"
BENCHMARK_SPLIT = "test"
DIFFICULTY_METRIC = "mean_normalized_entropy"
BENCHMARK_FUZZY_THRESHOLD = 95.0
ROWS_PER_CELL = 33
TRAIN_ROWS_PER_CELL = 30
VALIDATION_ROWS_PER_CELL = 3
RANDOM_SEED = 42

PUSH_TO_HUB = True
HF_DATASET_REPO_ID = None  # None -> <HF_TOKEN username>/<HF_DATASET_REPO_NAME>
HF_DATASET_REPO_NAME = "Korean-Olympiad-TDCS-3K"
HF_PRIVATE = False
DOWNLOAD_FULL_PARQUETS = False

## 1. Load and join the candidate rows with entropy metrics

In [ ]:
import json
import re
import unicodedata
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict, load_dataset

output_root = Path(OUTPUT_ROOT)
output_root.mkdir(parents=True, exist_ok=True)
TRAIN_PARQUET_PATH = output_root / "train.parquet"
VALIDATION_PARQUET_PATH = output_root / "validation.parquet"
ALL_PARQUET_PATH = output_root / "all_3300.parquet"
EXCLUSION_AUDIT_PATH = output_root / "benchmark_fuzzy_exclusion_audit.csv"
CELL_COUNTS_PATH = output_root / "cell_counts.csv"
SUMMARY_PATH = output_root / "dataset_summary.json"
README_PATH = output_root / "README.md"
DOWNLOAD_BUNDLE_PATH = output_root / "final_dataset_summary_outputs.zip"

candidate_path = Path(CANDIDATES_PATH)
metrics_path = Path(ENTROPY_METRICS_PATH)
if not candidate_path.exists() or not metrics_path.exists():
    raise FileNotFoundError("Run the entropy-inspection notebook first; candidate or metric file is missing")

candidate_dataset = load_dataset("parquet", data_files=str(candidate_path), split="train")
candidate_frame = candidate_dataset.to_pandas()
metric_frame = pd.read_csv(metrics_path)
if DIFFICULTY_METRIC not in metric_frame.columns:
    raise ValueError(f"Missing difficulty metric: {DIFFICULTY_METRIC}")

metric_columns = [
    "candidate_id", "generated_tokens", "hit_token_limit",
    "mean_token_entropy_nats", "mean_normalized_entropy",
    "tail_token_entropy_nats", "mean_greedy_surprisal_nats",
]
frame = candidate_frame.merge(metric_frame[metric_columns], on="candidate_id", how="inner", validate="one_to_one")
if len(frame) != 5000:
    raise ValueError(f"Expected 5,000 joined candidates, found {len(frame):,}")
print(f"Joined candidates: {len(frame):,}")

## 2. Assign five entropy levels inside each cluster

Ranking and cutting independently inside every cluster guarantees 50 candidates in each cluster × difficulty cell before benchmark decontamination. Lower entropy is D1; higher entropy is D5.

In [ ]:
frame["difficulty_level"] = 0
for cluster_id, indices in frame.groupby("cluster_id", sort=True).groups.items():
    ranked = frame.loc[indices, DIFFICULTY_METRIC].rank(method="first")
    levels = pd.qcut(ranked, q=5, labels=False) + 1
    frame.loc[indices, "difficulty_level"] = levels.astype(int)
frame["difficulty_level"] = frame["difficulty_level"].astype(int)
frame["difficulty_label"] = "D" + frame["difficulty_level"].astype(str)
frame["cell_id"] = frame.apply(
    lambda row: f"C{int(row['cluster_id']):02d}_D{int(row['difficulty_level'])}", axis=1
)
initial_counts = frame.groupby(["cluster_id", "difficulty_level"]).size()
if len(initial_counts) != 100 or not (initial_counts == 50).all():
    raise AssertionError("Expected exactly 50 candidates in each of 100 cells")
print("Difficulty assignment complete: 100 cells × 50 candidates")

## 3. Remove benchmark-like rows at 95% fuzzy similarity

The original English text is compared because it is less affected by differences between Korean translations. Both normalized character similarity and raw weighted similarity are checked.

In [ ]:
from rapidfuzz import fuzz, process
from tqdm.auto import tqdm

def normalize_math_text(value):
    text = unicodedata.normalize("NFKC", str(value or "")).lower()
    text = re.sub(r"\\(?:left|right|quad|qquad|,|!|;|:)", "", text)
    text = text.replace("$", "")
    return re.sub(r"\s+", "", text)

benchmark = load_dataset(
    BENCHMARK_DATASET_ID, split=BENCHMARK_SPLIT, token=HF_TOKEN
)
benchmark_raw = [str(text or "") for text in benchmark["question"]]
benchmark_normalized = [normalize_math_text(text) for text in benchmark_raw]

match_rows = []
excluded_candidate_ids = set()
for row in tqdm(frame.itertuples(index=False), total=len(frame), desc="Checking benchmark similarity"):
    raw_problem = str(row.problem or "")
    normalized_problem = normalize_math_text(raw_problem)
    normalized_match = process.extractOne(
        normalized_problem, benchmark_normalized, scorer=fuzz.ratio
    )
    raw_match = process.extractOne(raw_problem, benchmark_raw, scorer=fuzz.WRatio)
    choices = [match for match in [normalized_match, raw_match] if match is not None]
    best_match = max(choices, key=lambda match: float(match[1]))
    score = float(best_match[1])
    benchmark_index = int(best_match[2])
    remove = score >= BENCHMARK_FUZZY_THRESHOLD
    if remove:
        excluded_candidate_ids.add(int(row.candidate_id))
    match_rows.append({
        "candidate_id": int(row.candidate_id),
        "original_dataset_index": int(row.original_dataset_index),
        "cluster_id": int(row.cluster_id),
        "difficulty_level": int(row.difficulty_level),
        "maximum_fuzzy_score": score,
        "benchmark_index": benchmark_index,
        "excluded": remove,
        "training_problem": raw_problem,
        "benchmark_problem": benchmark_raw[benchmark_index],
    })

match_audit = pd.DataFrame(match_rows).sort_values("maximum_fuzzy_score", ascending=False)
match_audit.to_csv(EXCLUSION_AUDIT_PATH, index=False)
eligible = frame[~frame["candidate_id"].isin(excluded_candidate_ids)].copy()
print(f"Benchmark-like rows removed: {len(excluded_candidate_ids):,}")
print(f"Eligible candidates remaining: {len(eligible):,}")
display(match_audit.head(20))

## 4. Select 33 rows per cell and split them 30/3

In [ ]:
eligible_counts = (
    eligible.groupby(["cluster_id", "difficulty_level"]).size().rename("eligible_rows").reset_index()
)
if len(eligible_counts) != 100:
    raise ValueError("At least one cluster × difficulty cell disappeared after exclusion")
insufficient = eligible_counts[eligible_counts["eligible_rows"] < ROWS_PER_CELL]
if not insufficient.empty:
    display(insufficient)
    raise ValueError("Some cells have fewer than 33 eligible rows; inspect the table above")

rng = np.random.default_rng(RANDOM_SEED)
selected_indices = []
validation_indices = []
for _, group in eligible.groupby(["cluster_id", "difficulty_level"], sort=True):
    chosen = rng.choice(group.index.to_numpy(), size=ROWS_PER_CELL, replace=False)
    validation = rng.choice(chosen, size=VALIDATION_ROWS_PER_CELL, replace=False)
    selected_indices.extend(chosen.tolist())
    validation_indices.extend(validation.tolist())

selected = eligible.loc[selected_indices].copy()
validation_index_set = set(validation_indices)
selected["split"] = [
    "validation" if index in validation_index_set else "train" for index in selected.index
]
train_frame = selected[selected["split"] == "train"].copy()
validation_frame = selected[selected["split"] == "validation"].copy()

if len(selected) != 3300 or len(train_frame) != 3000 or len(validation_frame) != 300:
    raise AssertionError(
        f"Unexpected sizes: all={len(selected)}, train={len(train_frame)}, validation={len(validation_frame)}"
    )
for split_name, split_frame, expected_per_cell in [
    ("train", train_frame, TRAIN_ROWS_PER_CELL),
    ("validation", validation_frame, VALIDATION_ROWS_PER_CELL),
]:
    counts = split_frame.groupby(["cluster_id", "difficulty_level"]).size()
    if len(counts) != 100 or not (counts == expected_per_cell).all():
        raise AssertionError(f"{split_name} is not balanced at {expected_per_cell} rows per cell")

cell_counts = eligible_counts.copy()
cell_counts["selected_rows"] = ROWS_PER_CELL
cell_counts["train_rows"] = TRAIN_ROWS_PER_CELL
cell_counts["validation_rows"] = VALIDATION_ROWS_PER_CELL
cell_counts.to_csv(CELL_COUNTS_PATH, index=False)
print("Final sizes: train=3,000; validation=300; total=3,300")

## 5. Save the final DatasetDict to Google Drive

In [ ]:
helper_columns = ["split"]
train_export = train_frame.drop(columns=helper_columns).sort_values(
    ["cluster_id", "difficulty_level", "candidate_id"]
).reset_index(drop=True)
validation_export = validation_frame.drop(columns=helper_columns).sort_values(
    ["cluster_id", "difficulty_level", "candidate_id"]
).reset_index(drop=True)
all_export = selected.sort_values(
    ["split", "cluster_id", "difficulty_level", "candidate_id"]
).reset_index(drop=True)

train_dataset = Dataset.from_pandas(train_export, preserve_index=False)
validation_dataset = Dataset.from_pandas(validation_export, preserve_index=False)
final_dataset = DatasetDict({"train": train_dataset, "validation": validation_dataset})
train_dataset.to_parquet(str(TRAIN_PARQUET_PATH))
validation_dataset.to_parquet(str(VALIDATION_PARQUET_PATH))
Dataset.from_pandas(all_export, preserve_index=False).to_parquet(str(ALL_PARQUET_PATH))
print(f"Saved train: {TRAIN_PARQUET_PATH}")
print(f"Saved validation: {VALIDATION_PARQUET_PATH}")

## 6. Create the dataset card and upload to Hugging Face

`HF_TOKEN` must have write permission. With `HF_DATASET_REPO_ID = None`, the notebook automatically uses the account associated with the token.

In [ ]:
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)
identity = api.whoami(token=HF_TOKEN, cache=True)
username = identity["name"]
repo_id = HF_DATASET_REPO_ID or f"{username}/{HF_DATASET_REPO_NAME}"

card_text = f"""---
license: cc-by-nc-4.0
language:
- ko
- en
task_categories:
- text-generation
pretty_name: Korean Olympiad TDCS 3K
---

# Korean Olympiad TDCS 3K

A balanced Korean Olympiad mathematics reasoning dataset derived from
[ChuGyouk/AI-MO-NuminaMath-CoT-Ko](https://huggingface.co/datasets/ChuGyouk/AI-MO-NuminaMath-CoT-Ko).

## Splits

- `train`: 3,000 rows (30 from every cluster × difficulty cell)
- `validation`: 300 rows (3 from every cluster × difficulty cell)

## Construction

- Filtered source: `olympiads`
- Problem embeddings: `Qwen/Qwen3-Embedding-4B`, 1,024 dimensions
- Problem-type clusters: 20 spherical k-means clusters
- Difficulty proxy: per-cluster quintiles of mean normalized next-token entropy from
  `LGAI-EXAONE/EXAONE-4.0-1.2B` over up to 512 generated reasoning tokens
- Benchmark decontamination: rows with at least {BENCHMARK_FUZZY_THRESHOLD:.0f}% fuzzy similarity to
  [OlympiadBench-Math-Ko](https://huggingface.co/datasets/ChuGyouk/OlympiadBench-Math-Ko) were excluded before sampling
- Random seed: {RANDOM_SEED}

## Important limitations

- Difficulty is model-relative uncertainty, not a human difficulty annotation.
- Most uncertainty generations reached the 512-token limit, so entropy measures the first 512 reasoning tokens rather than final-answer confidence.
- The upstream solutions are machine-generated translations/reasoning traces and may contain mathematical errors.
- Fuzzy matching reduces benchmark leakage but cannot guarantee complete decontamination.

## License

This derivative follows the upstream CC BY-NC 4.0 license.
"""
README_PATH.write_text(card_text, encoding="utf-8")

hub_url = None
if PUSH_TO_HUB:
    api.create_repo(
        repo_id=repo_id, repo_type="dataset", private=HF_PRIVATE, exist_ok=True, token=HF_TOKEN
    )
    final_dataset.push_to_hub(repo_id, private=HF_PRIVATE, token=HF_TOKEN)
    api.upload_file(
        path_or_fileobj=str(README_PATH),
        path_in_repo="README.md",
        repo_id=repo_id,
        repo_type="dataset",
        token=HF_TOKEN,
        commit_message="Add dataset card",
    )
    hub_url = f"https://huggingface.co/datasets/{repo_id}"
    print(f"Uploaded dataset: {hub_url}")
else:
    print(f"PUSH_TO_HUB is disabled. Intended repository: {repo_id}")

## 7. Save and download the final summary

In [ ]:
from google.colab import files

summary = {
    "source_candidates": len(frame),
    "benchmark_fuzzy_threshold": BENCHMARK_FUZZY_THRESHOLD,
    "benchmark_like_rows_removed": len(excluded_candidate_ids),
    "eligible_rows": len(eligible),
    "clusters": 20,
    "difficulty_levels": 5,
    "rows_per_cell": ROWS_PER_CELL,
    "train_rows_per_cell": TRAIN_ROWS_PER_CELL,
    "validation_rows_per_cell": VALIDATION_ROWS_PER_CELL,
    "train_rows": len(train_dataset),
    "validation_rows": len(validation_dataset),
    "total_rows": len(train_dataset) + len(validation_dataset),
    "random_seed": RANDOM_SEED,
    "hub_repo_id": repo_id,
    "hub_url": hub_url,
}
SUMMARY_PATH.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(summary, ensure_ascii=False, indent=2))

with zipfile.ZipFile(DOWNLOAD_BUNDLE_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in [SUMMARY_PATH, CELL_COUNTS_PATH, EXCLUSION_AUDIT_PATH, README_PATH]:
        archive.write(path, arcname=path.name)
files.download(str(DOWNLOAD_BUNDLE_PATH))

if DOWNLOAD_FULL_PARQUETS:
    files.download(str(TRAIN_PARQUET_PATH))
    files.download(str(VALIDATION_PARQUET_PATH))
else:
    print(f"Full Parquet files remain saved under: {output_root}")